In [1]:
!pip install opencv-python mediapipe==0.10.9 numpy

  Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached mediapipe-0.10.9-cp310-cp310-win_amd64.whl.metadata (9.8 kB)
  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl.metadata (60 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached opencv_contrib_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached protobuf-3.20.3-cp310-cp310-win_amd64.whl.metadata (698 bytes)
  Using cached sounddevice-0.5.5-py3-none-win_amd64.whl.metadata (1.4 kB)
  Using cached contourpy-1.3.2-cp310-cp310-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pillow-12.2.0-cp310-cp310-win_amd64.whl.metadata (9.0 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached mediapipe-0.10.9-cp310-cp310-win_amd64.whl (50.5 MB)
Using cached protobuf-3.20.3-cp310-cp310-win_amd64.

In [1]:
import mediapipe as mp
mp_face_mesh = mp.solutions.face_mesh

print("DONE ✅")

DONE ✅


In [2]:
!pip install transformers torch torchvision opencv-python pillow

   ---------------------------------------- 0.0/4.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.3 MB ? eta -:--:--
   ------- -------------------------------- 0.8/4.3 MB 3.9 MB/s eta 0:00:01
   -------------- ------------------------- 1.6/4.3 MB 4.0 MB/s eta 0:00:01
   --------------------- ------------------ 2.4/4.3 MB 3.9 MB/s eta 0:00:01
   ------------------------------- -------- 3.4/4.3 MB 3.9 MB/s eta 0:00:01
   -------------------------------------- - 4.2/4.3 MB 3.9 MB/s eta 0:00:01
   ---------------------------------------- 4.3/4.3 MB 3.9 MB/s  0:00:01


In [3]:

import sys
!{sys.executable} -m pip install pyserial


In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import serial
import serial.tools.list_ports
from collections import deque
import time

# ===== Bluetooth Setup =====
def connect_bluetooth():
    ports = serial.tools.list_ports.comports()
    print("📡 Available ports:")
    for p in ports:
        print(f"   {p.device} - {p.description}")
    
    bt_port = None
    for p in ports:
        if any(x in p.description.upper() for x in ["BLUETOOTH", "HC-05", "HC-06", "ESP32", "COM"]):
            bt_port = p.device
            break
    
    if bt_port is None:
        print("⚠️  No Bluetooth found — running without Bluetooth")
        return None
    
    try:
        bt = serial.Serial(bt_port, baudrate=9600, timeout=1)
        print(f"✅ Connected to {bt_port}")
        return bt
    except Exception as e:
        print(f"❌ Bluetooth error: {e}")
        return None

bt = connect_bluetooth()

# ===== MediaPipe =====
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(refine_landmarks=True, max_num_faces=1)
cap = cv2.VideoCapture(0)

# ===== Landmarks =====
LEFT_IRIS   = [474, 475, 476, 477]
RIGHT_IRIS  = [469, 470, 471, 472]
LEFT_EYE_H  = [33, 133]
RIGHT_EYE_H = [362, 263]
LEFT_EYE    = [33, 160, 158, 133, 153, 144]
RIGHT_EYE   = [362, 385, 387, 263, 373, 380]

# ===== Blink Settings =====
BLINK_THRESHOLD  = 0.20
BLINK_FRAMES     = 3
blink_counter    = 0
blink_cooldown   = 0

# ===== Control State =====
is_stopped       = False
close_counter    = 0
CLOSE_FRAMES     = 60          # ~2 ثانية عشان يعمل toggle

# ===== Smoothing =====
HISTORY_SIZE     = 10
direction_history = deque(maxlen=HISTORY_SIZE)

# ===== Bluetooth =====
last_command     = None
COMMAND_MAP      = {
    "FORWARD":  "F",
    "LEFT":     "L",
    "RIGHT":    "R",
    "STOP":     "S",
    "BLINKING": None,          # مش بيبعت حاجة
}

# ===== FPS =====
frame_count  = 0
start_time   = time.time()
fps          = 0

def eye_aspect_ratio(eye):
    A = np.linalg.norm(np.array(eye[1]) - np.array(eye[5]))
    B = np.linalg.norm(np.array(eye[2]) - np.array(eye[4]))
    C = np.linalg.norm(np.array(eye[0]) - np.array(eye[3]))
    return (A + B) / (2.0 * C)

def get_iris_ratio(face, iris_ids, corner_ids):
    iris_x  = np.mean([face.landmark[i].x for i in iris_ids])
    x1 = face.landmark[corner_ids[0]].x
    x2 = face.landmark[corner_ids[1]].x
    left_x  = min(x1, x2)
    right_x = max(x1, x2)
    eye_w   = max(right_x - left_x, 0.001)
    return (iris_x - left_x) / eye_w

def send_command(direction):
    global last_command
    cmd = COMMAND_MAP.get(direction)
    if cmd and cmd != last_command and bt:
        try:
            bt.write(cmd.encode())
            last_command = cmd
            print(f"📡 Sent: {cmd} ({direction})")
        except Exception as e:
            print(f"❌ BT Error: {e}")

def smooth_direction(history):
    if not history:
        return "FORWARD"
    counts = {}
    for d in history:
        counts[d] = counts.get(d, 0) + 1
    return max(counts, key=counts.get)

# ===== Main Loop =====
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb)
    h, w, _ = frame.shape

    # ===== FPS =====
    frame_count += 1
    elapsed = time.time() - start_time
    if elapsed >= 1.0:
        fps = frame_count / elapsed
        frame_count = 0
        start_time = time.time()

    direction = "FORWARD"

    # ===== No Face =====
    if not results.multi_face_landmarks:
        cv2.rectangle(frame, (0, 0), (w, h), (0, 0, 80), 4)
        cv2.rectangle(frame, (10, 10), (420, 70), (0, 0, 200), -1)
        cv2.putText(frame, "NO FACE DETECTED", (20, 55),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.1, (255, 255, 255), 3)
        send_command("STOP")
        cv2.imshow("EYEonic", frame)
        if cv2.waitKey(1) & 0xFF == 27:
            break
        continue

    face = results.multi_face_landmarks[0]

    # ===== Blink Cooldown =====
    if blink_cooldown > 0:
        blink_cooldown -= 1

    # ===== Eye Points =====
    left_eye, right_eye = [], []
    for idx in LEFT_EYE:
        x = int(face.landmark[idx].x * w)
        y = int(face.landmark[idx].y * h)
        left_eye.append((x, y))
    for idx in RIGHT_EYE:
        x = int(face.landmark[idx].x * w)
        y = int(face.landmark[idx].y * h)
        right_eye.append((x, y))

    ear = (eye_aspect_ratio(left_eye) + eye_aspect_ratio(right_eye)) / 2

    # ===== Toggle STOP (2 ثانية عين مقفولة) =====
    if ear < BLINK_THRESHOLD:
        close_counter += 1
        blink_counter += 1
    else:
        close_counter = 0
        if blink_counter >= BLINK_FRAMES and blink_cooldown == 0:
            pass  # الرمشة العادية مش بتعمل حاجة — التحكم بالـ 2 ثانية بس
        blink_counter = 0

    if close_counter >= CLOSE_FRAMES and blink_cooldown == 0:
        is_stopped = not is_stopped
        blink_cooldown = 30
        close_counter  = 0
        print(f"🚦 Control: {'STOP' if is_stopped else 'GO'}")

    # ===== Iris Ratio =====
    ratio = (get_iris_ratio(face, LEFT_IRIS, LEFT_EYE_H) +
             get_iris_ratio(face, RIGHT_IRIS, RIGHT_EYE_H)) / 2

    # ===== Raw Direction =====
    if is_stopped:
        raw_dir = "STOP"
    elif ear < BLINK_THRESHOLD:
        raw_dir = "BLINKING"
    elif ratio < 0.40:
        raw_dir = "LEFT"
    elif ratio > 0.60:
        raw_dir = "RIGHT"
    else:
        raw_dir = "FORWARD"

    # ===== Smoothing =====
    direction_history.append(raw_dir)
    direction = smooth_direction(direction_history)

    # ===== Send Bluetooth =====
    send_command(direction)

    # ===== Draw Iris =====
    for idx in LEFT_IRIS + RIGHT_IRIS:
        x = int(face.landmark[idx].x * w)
        y = int(face.landmark[idx].y * h)
        cv2.circle(frame, (x, y), 3, (0, 255, 0), -1)

    for p in left_eye + right_eye:
        cv2.circle(frame, p, 1, (255, 0, 0), -1)

    # ===== Side Panel =====
    panel_w = 280
    panel = np.ones((h, panel_w, 3), dtype=np.uint8) * 30

    colors = {
        "FORWARD":  (0, 180, 0),
        "LEFT":     (0, 140, 255),
        "RIGHT":    (0, 140, 255),
        "STOP":     (0, 0, 220),
        "BLINKING": (180, 0, 180),
    }
    box_color = colors.get(direction, (100, 100, 100))

    # Control status
    ctrl_color = (0, 200, 0) if not is_stopped else (0, 0, 220)
    ctrl_text  = "CONTROL: ON" if not is_stopped else "CONTROL: OFF"
    cv2.rectangle(panel, (10, 10), (panel_w-10, 60), ctrl_color, -1)
    cv2.putText(panel, ctrl_text, (15, 45),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)

    # Direction
    cv2.rectangle(panel, (10, 80), (panel_w-10, 150), box_color, -1)
    cv2.putText(panel, direction, (15, 133),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255,255,255), 3)

    # Arrow
    cx, cy = panel_w // 2, 210
    if direction == "FORWARD":
        cv2.arrowedLine(panel, (cx, cy+40), (cx, cy-40), (0,200,0), 4, tipLength=0.4)
    elif direction == "LEFT":
        cv2.arrowedLine(panel, (cx+40, cy), (cx-40, cy), (0,140,255), 4, tipLength=0.4)
    elif direction == "RIGHT":
        cv2.arrowedLine(panel, (cx-40, cy), (cx+40, cy), (0,140,255), 4, tipLength=0.4)
    elif direction == "STOP":
        cv2.rectangle(panel, (cx-30, cy-30), (cx+30, cy+30), (0,0,220), -1)
        cv2.putText(panel, "||", (cx-18, cy+12),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255,255,255), 3)
    elif direction == "BLINKING":
        cv2.putText(panel, "* *", (cx-30, cy+10),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (180,0,180), 3)

    # Stats
    bt_status = "🔵 BT: ON" if bt else "🔴 BT: OFF"
    cv2.putText(panel, bt_status,        (10, h-100), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 1)
    cv2.putText(panel, f"FPS: {fps:.1f}", (10, h-70),  cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 1)
    cv2.putText(panel, f"EAR: {ear:.2f}", (10, h-45),  cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,0),   1)
    cv2.putText(panel, f"ratio:{ratio:.2f}",(10, h-20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,0),   1)

    # ===== Merge =====
    combined = np.hstack((frame, panel))
    cv2.imshow("EYEonic", combined)

    if cv2.waitKey(1) & 0xFF == 27:
        break

# ===== Cleanup =====
cap.release()
cv2.destroyAllWindows()
if bt:
    bt.close()
    print("🔚 Bluetooth closed.")

In [12]:
import cv2
cap = cv2.VideoCapture(0)
ret, frame = cap.read()
print(ret)
cap.release()

True


import cv2
import mediapipe as mp
import numpy as np
import serial
import serial.tools.list_ports
from collections import deque
import time

# ===== Bluetooth Setup =====
def connect_bluetooth():
    ports = serial.tools.list_ports.comports()
    print("📡 Available ports:")
    for p in ports:
        print(f"   {p.device} - {p.description}")

    bt_port = None
    for p in ports:
        if "BLUETOOTH" in p.description.upper():
            bt_port = p.device
            break

    if bt_port is None:
        print("⚠️  No Bluetooth found — running without Bluetooth")
        return None

    try:
        bt = serial.Serial(bt_port, baudrate=9600, timeout=0.1)
        print(f"✅ Connected to {bt_port}")
        return bt
    except Exception as e:
        print(f"❌ Bluetooth error: {e}")
        return None

# ===== Camera First =====
cap = cv2.VideoCapture(0)
print("📷 Camera ready:", cap.isOpened())

# ===== Bluetooth After =====
bt = connect_bluetooth()

# ===== MediaPipe =====
mp_face_mesh = mp.solutions.face_mesh
face_mesh    = mp_face_mesh.FaceMesh(refine_landmarks=True, max_num_faces=1)

# ===== Landmarks =====
LEFT_IRIS   = [474, 475, 476, 477]
RIGHT_IRIS  = [469, 470, 471, 472]
LEFT_EYE_H  = [33, 133]
RIGHT_EYE_H = [362, 263]
LEFT_EYE    = [33, 160, 158, 133, 153, 144]
RIGHT_EYE   = [362, 385, 387, 263, 373, 380]

# ===== Settings =====
BLINK_THRESHOLD   = 0.20
blink_cooldown    = 0
close_counter     = 0
CLOSE_FRAMES      = 60
is_stopped        = False
direction_history = deque(maxlen=10)
last_command      = None
frame_count       = 0
start_time        = time.time()
fps               = 0

COMMAND_MAP = {
    "FORWARD":  "F",
    "LEFT":     "L",
    "RIGHT":    "R",
    "STOP":     "S",
    "BLINKING": "S",
}

# ===== Functions =====
def eye_aspect_ratio(eye):
    A = np.linalg.norm(np.array(eye[1]) - np.array(eye[5]))
    B = np.linalg.norm(np.array(eye[2]) - np.array(eye[4]))
    C = np.linalg.norm(np.array(eye[0]) - np.array(eye[3]))
    return (A + B) / (2.0 * C)

def get_iris_ratio(face, iris_ids, corner_ids):
    iris_x  = np.mean([face.landmark[i].x for i in iris_ids])
    x1      = face.landmark[corner_ids[0]].x
    x2      = face.landmark[corner_ids[1]].x
    left_x  = min(x1, x2)
    right_x = max(x1, x2)
    eye_w   = max(right_x - left_x, 0.001)
    return (iris_x - left_x) / eye_w

def send_command(direction):
    global last_command
    cmd = COMMAND_MAP.get(direction)
    if cmd and cmd != last_command and bt:
        try:
            bt.write(cmd.encode())
            last_command = cmd
            print(f"📡 Sent: {cmd} ({direction})")
        except Exception as e:
            print(f"❌ BT Error: {e}")

def smooth_direction(history):
    if not history:
        return "FORWARD"
    counts = {}
    for d in history:
        counts[d] = counts.get(d, 0) + 1
    return max(counts, key=counts.get)

# ===== Main Loop =====
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb)
    h, w, _ = frame.shape

    # FPS
    frame_count += 1
    elapsed = time.time() - start_time
    if elapsed >= 1.0:
        fps         = frame_count / elapsed
        frame_count = 0
        start_time  = time.time()

    direction = "FORWARD"

    # ===== No Face =====
    if not results.multi_face_landmarks:
        cv2.rectangle(frame, (0, 0), (w, h), (0, 0, 80), 4)
        cv2.rectangle(frame, (10, 10), (420, 70), (0, 0, 200), -1)
        cv2.putText(frame, "NO FACE DETECTED", (20, 55),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.1, (255, 255, 255), 3)
        send_command("STOP")
        cv2.imshow("EYEonic", frame)
        if cv2.waitKey(1) & 0xFF == 27:
            break
        continue

    face = results.multi_face_landmarks[0]

    if blink_cooldown > 0:
        blink_cooldown -= 1

    # ===== Eye Points =====
    left_eye, right_eye = [], []
    for idx in LEFT_EYE:
        x = int(face.landmark[idx].x * w)
        y = int(face.landmark[idx].y * h)
        left_eye.append((x, y))
    for idx in RIGHT_EYE:
        x = int(face.landmark[idx].x * w)
        y = int(face.landmark[idx].y * h)
        right_eye.append((x, y))

    ear = (eye_aspect_ratio(left_eye) + eye_aspect_ratio(right_eye)) / 2

    # ===== Toggle STOP =====
    if ear < BLINK_THRESHOLD:
        close_counter += 1
    else:
        close_counter = 0

    if close_counter >= CLOSE_FRAMES and blink_cooldown == 0:
        is_stopped     = not is_stopped
        blink_cooldown = 40
        close_counter  = 0
        print(f"🚦 Control: {'STOP' if is_stopped else 'GO'}")

    # ===== Iris Ratio =====
    ratio = (get_iris_ratio(face, LEFT_IRIS, LEFT_EYE_H) +
             get_iris_ratio(face, RIGHT_IRIS, RIGHT_EYE_H)) / 2

    # ===== Direction =====
    if is_stopped:
        raw_dir = "STOP"
    elif ear < BLINK_THRESHOLD:
        raw_dir = "BLINKING"
    elif ratio < 0.40:
        raw_dir = "LEFT"
    elif ratio > 0.60:
        raw_dir = "RIGHT"
    else:
        raw_dir = "FORWARD"

    direction_history.append(raw_dir)
    direction = smooth_direction(direction_history)
    send_command(direction)

    # ===== Draw =====
    for idx in LEFT_IRIS + RIGHT_IRIS:
        x = int(face.landmark[idx].x * w)
        y = int(face.landmark[idx].y * h)
        cv2.circle(frame, (x, y), 3, (0, 255, 0), -1)

    for p in left_eye + right_eye:
        cv2.circle(frame, p, 1, (255, 0, 0), -1)

    # ===== Side Panel =====
    panel_w = 280
    panel   = np.ones((h, panel_w, 3), dtype=np.uint8) * 30

    colors = {
        "FORWARD":  (0, 180, 0),
        "LEFT":     (0, 140, 255),
        "RIGHT":    (0, 140, 255),
        "STOP":     (0, 0, 220),
        "BLINKING": (180, 0, 180),
    }
    box_color  = colors.get(direction, (100, 100, 100))
    ctrl_color = (0, 200, 0) if not is_stopped else (0, 0, 220)
    ctrl_text  = "CONTROL: ON" if not is_stopped else "CONTROL: OFF"

    cv2.rectangle(panel, (10, 10), (panel_w-10, 60), ctrl_color, -1)
    cv2.putText(panel, ctrl_text, (15, 45),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    cv2.rectangle(panel, (10, 80), (panel_w-10, 150), box_color, -1)
    cv2.putText(panel, direction, (15, 133),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3)

    cx, cy = panel_w // 2, 220
    if direction == "FORWARD":
        cv2.arrowedLine(panel, (cx, cy+40), (cx, cy-40), (0, 200, 0), 4, tipLength=0.4)
    elif direction == "LEFT":
        cv2.arrowedLine(panel, (cx+40, cy), (cx-40, cy), (0, 140, 255), 4, tipLength=0.4)
    elif direction == "RIGHT":
        cv2.arrowedLine(panel, (cx-40, cy), (cx+40, cy), (0, 140, 255), 4, tipLength=0.4)
    elif direction == "STOP":
        cv2.rectangle(panel, (cx-30, cy-30), (cx+30, cy+30), (0, 0, 220), -1)
        cv2.putText(panel, "||", (cx-18, cy+12),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3)
    elif direction == "BLINKING":
        cv2.putText(panel, "* *", (cx-30, cy+10),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (180, 0, 180), 3)

    bt_str = "BT: ON" if bt else "BT: OFF"
    bt_col = (0, 200, 0) if bt else (0, 0, 200)
    cv2.putText(panel, bt_str,               (10, h-100), cv2.FONT_HERSHEY_SIMPLEX, 0.6, bt_col, 2)
    cv2.putText(panel, f"FPS:   {fps:.1f}",  (10, h-75),  cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
    cv2.putText(panel, f"EAR:   {ear:.2f}",  (10, h-50),  cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 1)
    cv2.putText(panel, f"ratio: {ratio:.2f}", (10, h-25),  cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 1)

    combined = np.hstack((frame, panel))
    cv2.imshow("EYEonic", combined)

    if cv2.waitKey(1) & 0xFF == 27:
        break

# ===== Cleanup =====
cap.release()
cv2.destroyAllWindows()
if bt:
    bt.close()
    print("🔚 Bluetooth closed.")

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import serial
import serial.tools.list_ports
from collections import deque
import time

# ===== Bluetooth Setup =====
def connect_bluetooth():
    ports = serial.tools.list_ports.comports()
    print("📡 Available ports:")
    for p in ports:
        print(f"   {p.device} - {p.description}")

    bt_port = None
    for p in ports:
        if "BLUETOOTH" in p.description.upper():
            bt_port = p.device
            break

    if bt_port is None:
        print("⚠️  No Bluetooth found — running without Bluetooth")
        return None

    try:
        bt = serial.Serial(bt_port, baudrate=9600, timeout=0.1)
        print(f"✅ Connected to {bt_port}")
        return bt
    except Exception as e:
        print(f"❌ Bluetooth error: {e}")
        return None

# ===== Camera First =====
cap = cv2.VideoCapture(0)
print("📷 Camera ready:", cap.isOpened())

# ===== Bluetooth After =====
bt = connect_bluetooth()

# ===== MediaPipe =====
mp_face_mesh = mp.solutions.face_mesh
face_mesh    = mp_face_mesh.FaceMesh(refine_landmarks=True, max_num_faces=1)

# ===== Landmarks =====
LEFT_IRIS   = [474, 475, 476, 477]
RIGHT_IRIS  = [469, 470, 471, 472]
LEFT_EYE_H  = [33, 133]
RIGHT_EYE_H = [362, 263]
LEFT_EYE    = [33, 160, 158, 133, 153, 144]
RIGHT_EYE   = [362, 385, 387, 263, 373, 380]

# ===== Settings =====
BLINK_THRESHOLD   = 0.20
blink_cooldown    = 0
close_counter     = 0
CLOSE_FRAMES      = 60
is_stopped        = False
direction_history = deque(maxlen=10)
last_command      = None
frame_count       = 0
start_time        = time.time()
fps               = 0

COMMAND_MAP = {
    "FORWARD":  "F",
    "LEFT":     "L",
    "RIGHT":    "R",
    "STOP":     "S",
    "BLINKING": "S",
}

# ===== Functions =====
def eye_aspect_ratio(eye):
    A = np.linalg.norm(np.array(eye[1]) - np.array(eye[5]))
    B = np.linalg.norm(np.array(eye[2]) - np.array(eye[4]))
    C = np.linalg.norm(np.array(eye[0]) - np.array(eye[3]))
    return (A + B) / (2.0 * C)

def get_iris_ratio(face, iris_ids, corner_ids):
    iris_x  = np.mean([face.landmark[i].x for i in iris_ids])
    x1      = face.landmark[corner_ids[0]].x
    x2      = face.landmark[corner_ids[1]].x
    left_x  = min(x1, x2)
    right_x = max(x1, x2)
    eye_w   = max(right_x - left_x, 0.001)
    return (iris_x - left_x) / eye_w

def send_command(direction):
    global last_command
    cmd = COMMAND_MAP.get(direction)
    if cmd and cmd != last_command and bt:
        try:
            bt.write(cmd.encode())
            last_command = cmd
            print(f"📡 Sent: {cmd} ({direction})")
        except Exception as e:
            print(f"❌ BT Error: {e}")

def smooth_direction(history):
    if not history:
        return "FORWARD"
    counts = {}
    for d in history:
        counts[d] = counts.get(d, 0) + 1
    return max(counts, key=counts.get)

# ===== Main Loop =====
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb)
    h, w, _ = frame.shape

    # FPS
    frame_count += 1
    elapsed = time.time() - start_time
    if elapsed >= 1.0:
        fps         = frame_count / elapsed
        frame_count = 0
        start_time  = time.time()

    direction = "FORWARD"

    # ===== No Face =====
    if not results.multi_face_landmarks:
        cv2.rectangle(frame, (0, 0), (w, h), (0, 0, 80), 4)
        cv2.rectangle(frame, (10, 10), (420, 70), (0, 0, 200), -1)
        cv2.putText(frame, "NO FACE DETECTED", (20, 55),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.1, (255, 255, 255), 3)
        send_command("STOP")
        cv2.imshow("EYEonic", frame)
        if cv2.waitKey(1) & 0xFF == 27:
            break
        continue

    face = results.multi_face_landmarks[0]

    if blink_cooldown > 0:
        blink_cooldown -= 1

    # ===== Eye Points =====
    left_eye, right_eye = [], []
    for idx in LEFT_EYE:
        x = int(face.landmark[idx].x * w)
        y = int(face.landmark[idx].y * h)
        left_eye.append((x, y))
    for idx in RIGHT_EYE:
        x = int(face.landmark[idx].x * w)
        y = int(face.landmark[idx].y * h)
        right_eye.append((x, y))

    ear = (eye_aspect_ratio(left_eye) + eye_aspect_ratio(right_eye)) / 2

    # ===== Toggle STOP =====
    if ear < BLINK_THRESHOLD:
        close_counter += 1
    else:
        close_counter = 0

    if close_counter >= CLOSE_FRAMES and blink_cooldown == 0:
        is_stopped     = not is_stopped
        blink_cooldown = 40
        close_counter  = 0
        print(f"🚦 Control: {'STOP' if is_stopped else 'GO'}")

    # ===== Iris Ratio =====
    ratio = (get_iris_ratio(face, LEFT_IRIS, LEFT_EYE_H) +
             get_iris_ratio(face, RIGHT_IRIS, RIGHT_EYE_H)) / 2

    # ===== Direction =====
    if is_stopped:
        raw_dir = "STOP"
    elif ear < BLINK_THRESHOLD:
        raw_dir = "BLINKING"
    elif ratio < 0.40:
        raw_dir = "LEFT"
    elif ratio > 0.60:
        raw_dir = "RIGHT"
    else:
        raw_dir = "FORWARD"

    direction_history.append(raw_dir)
    direction = smooth_direction(direction_history)
    send_command(direction)

    # ===== Draw =====
    for idx in LEFT_IRIS + RIGHT_IRIS:
        x = int(face.landmark[idx].x * w)
        y = int(face.landmark[idx].y * h)
        cv2.circle(frame, (x, y), 3, (0, 255, 0), -1)

    for p in left_eye + right_eye:
        cv2.circle(frame, p, 1, (255, 0, 0), -1)

    # ===== Side Panel =====
    panel_w = 280
    panel   = np.ones((h, panel_w, 3), dtype=np.uint8) * 30

    colors = {
        "FORWARD":  (0, 180, 0),
        "LEFT":     (0, 140, 255),
        "RIGHT":    (0, 140, 255),
        "STOP":     (0, 0, 220),
        "BLINKING": (180, 0, 180),
    }
    box_color  = colors.get(direction, (100, 100, 100))
    ctrl_color = (0, 200, 0) if not is_stopped else (0, 0, 220)
    ctrl_text  = "CONTROL: ON" if not is_stopped else "CONTROL: OFF"

    cv2.rectangle(panel, (10, 10), (panel_w-10, 60), ctrl_color, -1)
    cv2.putText(panel, ctrl_text, (15, 45),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    cv2.rectangle(panel, (10, 80), (panel_w-10, 150), box_color, -1)
    cv2.putText(panel, direction, (15, 133),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3)

    cx, cy = panel_w // 2, 220
    if direction == "FORWARD":
        cv2.arrowedLine(panel, (cx, cy+40), (cx, cy-40), (0, 200, 0), 4, tipLength=0.4)
    elif direction == "LEFT":
        cv2.arrowedLine(panel, (cx+40, cy), (cx-40, cy), (0, 140, 255), 4, tipLength=0.4)
    elif direction == "RIGHT":
        cv2.arrowedLine(panel, (cx-40, cy), (cx+40, cy), (0, 140, 255), 4, tipLength=0.4)
    elif direction == "STOP":
        cv2.rectangle(panel, (cx-30, cy-30), (cx+30, cy+30), (0, 0, 220), -1)
        cv2.putText(panel, "||", (cx-18, cy+12),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3)
    elif direction == "BLINKING":
        cv2.putText(panel, "* *", (cx-30, cy+10),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (180, 0, 180), 3)

    bt_str = "BT: ON" if bt else "BT: OFF"
    bt_col = (0, 200, 0) if bt else (0, 0, 200)
    cv2.putText(panel, bt_str,               (10, h-100), cv2.FONT_HERSHEY_SIMPLEX, 0.6, bt_col, 2)
    cv2.putText(panel, f"FPS:   {fps:.1f}",  (10, h-75),  cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
    cv2.putText(panel, f"EAR:   {ear:.2f}",  (10, h-50),  cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 1)
    cv2.putText(panel, f"ratio: {ratio:.2f}", (10, h-25),  cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 1)

    combined = np.hstack((frame, panel))
    cv2.imshow("EYEonic", combined)

    if cv2.waitKey(1) & 0xFF == 27:
        break

# ===== Cleanup =====
cap.release()
cv2.destroyAllWindows()
if bt:
    bt.close()
    print("🔚 Bluetooth closed.")

📷 Camera ready: True
📡 Available ports:
   COM4 - Standard Serial over Bluetooth link (COM4)
   COM6 - Standard Serial over Bluetooth link (COM6)
   COM3 - Silicon Labs CP210x USB to UART Bridge (COM3)
   COM7 - Standard Serial over Bluetooth link (COM7)
   COM5 - Standard Serial over Bluetooth link (COM5)
✅ Connected to COM4


In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import serial
import serial.tools.list_ports
from collections import deque
import time

# ===== Bluetooth Setup =====
def connect_bluetooth():
    ports = serial.tools.list_ports.comports()
    print(" Available ports:")
    for p in ports:
        print(f"   {p.device} - {p.description}")

    bt_port = None
    for p in ports:
        if "BLUETOOTH" in p.description.upper():
            bt_port = p.device
            break

    if bt_port is None:
        print(" No Bluetooth found — running without Bluetooth")
        return None

    try:
        bt = serial.Serial(bt_port, baudrate=9600, timeout=1)
        print(f"Connected to {bt_port}")
        return bt
    except Exception as e:
        print(f" Bluetooth error: {e}")
        return None

bt = None

try:
    bt = serial.Serial('COM7', 115200, timeout=1)
    print("✅ Connected to ESP32")
except Exception as e:
    print("❌ BT Error:", e)

# ===== MediaPipe =====
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(refine_landmarks=True, max_num_faces=1)
cap = cv2.VideoCapture(0)

# ===== Landmarks =====
LEFT_IRIS   = [474, 475, 476, 477]
RIGHT_IRIS  = [469, 470, 471, 472]
LEFT_EYE_H  = [33, 133]
RIGHT_EYE_H = [362, 263]
LEFT_EYE    = [33, 160, 158, 133, 153, 144]
RIGHT_EYE   = [362, 385, 387, 263, 373, 380]

# ===== Blink Settings =====
BLINK_THRESHOLD = 0.20
blink_cooldown  = 0
close_counter   = 0
CLOSE_FRAMES    = 60

# ===== Control State =====
# ===== Control State =====
is_stopped = False
close_counter = 0

# مدة غلق العين المطلوبة لتفعيل Start/Stop
CLOSE_FRAMES = 60

# Emergency
eye_closed_start = None
EMERGENCY_FRAMES = 450
emergency_sent = False
# ===== Smoothing =====
HISTORY_SIZE      = 10
direction_history = deque(maxlen=HISTORY_SIZE)

# ===== Bluetooth =====
last_command = None
COMMAND_MAP  = {
    "FORWARD":  "F",
    "LEFT":     "L",
    "RIGHT":    "R",
    "STOP":     "S",
    "BLINKING": "S",
}

# ===== FPS =====
frame_count = 0
start_time  = time.time()
fps         = 0

# ===== Functions =====
def eye_aspect_ratio(eye):
    A = np.linalg.norm(np.array(eye[1]) - np.array(eye[5]))
    B = np.linalg.norm(np.array(eye[2]) - np.array(eye[4]))
    C = np.linalg.norm(np.array(eye[0]) - np.array(eye[3]))
    return (A + B) / (2.0 * C)

def get_iris_ratio(face, iris_ids, corner_ids):
    iris_x  = np.mean([face.landmark[i].x for i in iris_ids])
    x1      = face.landmark[corner_ids[0]].x
    x2      = face.landmark[corner_ids[1]].x
    left_x  = min(x1, x2)
    right_x = max(x1, x2)
    eye_w   = max(right_x - left_x, 0.001)
    return (iris_x - left_x) / eye_w

def send_command(direction):
    global last_command
    cmd = COMMAND_MAP.get(direction)
    if cmd and cmd != last_command and bt:
        try:
            bt.write(cmd.encode())
            last_command = cmd
            print(f" Sent: {cmd} ({direction})")
        except Exception as e:
            print(f" BT Error: {e}")

def smooth_direction(history):
    if not history:
        return "FORWARD"
    counts = {}
    for d in history:
        counts[d] = counts.get(d, 0) + 1
    return max(counts, key=counts.get)

# ===== Main Loop =====
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb)
    h, w, _ = frame.shape

    # ===== FPS =====
    frame_count += 1
    elapsed = time.time() - start_time
    if elapsed >= 1.0:
        fps         = frame_count / elapsed
        frame_count = 0
        start_time  = time.time()

    direction = "FORWARD"

    # ===== No Face =====
    if not results.multi_face_landmarks:
        cv2.rectangle(frame, (0, 0), (w, h), (0, 0, 80), 4)
        cv2.rectangle(frame, (10, 10), (420, 70), (0, 0, 200), -1)
        cv2.putText(frame, "NO FACE DETECTED", (20, 55),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.1, (255, 255, 255), 3)
        send_command("STOP")
        cv2.imshow("EYEonic", frame)
        if cv2.waitKey(1) & 0xFF == 27:
            break
        continue

    face = results.multi_face_landmarks[0]

    # ===== Cooldown =====
    if blink_cooldown > 0:
        blink_cooldown -= 1

    # ===== Eye Points =====
    left_eye, right_eye = [], []
    for idx in LEFT_EYE:
        x = int(face.landmark[idx].x * w)
        y = int(face.landmark[idx].y * h)
        left_eye.append((x, y))
    for idx in RIGHT_EYE:
        x = int(face.landmark[idx].x * w)
        y = int(face.landmark[idx].y * h)
        right_eye.append((x, y))

    ear = (eye_aspect_ratio(left_eye) + eye_aspect_ratio(right_eye)) / 2

    # ===== Toggle STOP =====
    if ear < BLINK_THRESHOLD:
        close_counter += 1
    else:
        close_counter = 0

    if close_counter >= CLOSE_FRAMES and blink_cooldown == 0:
        is_stopped     = not is_stopped
        blink_cooldown = 40
        close_counter  = 0
        print(f" Control: {'STOP' if is_stopped else 'GO'}")

    # ===== Iris Ratio =====
    ratio = (get_iris_ratio(face, LEFT_IRIS, LEFT_EYE_H) +
             get_iris_ratio(face, RIGHT_IRIS, RIGHT_EYE_H)) / 2

    # ===== Raw Direction =====
    if is_stopped:
        raw_dir = "STOP"
    elif ear < BLINK_THRESHOLD:
        raw_dir = "BLINKING"
    elif ratio < 0.40:
        raw_dir = "LEFT"
    elif ratio > 0.60:
        raw_dir = "RIGHT"
    else:
        raw_dir = "FORWARD"

    # ===== Smoothing =====
    direction_history.append(raw_dir)
    direction = smooth_direction(direction_history)

    # ===== Send Bluetooth =====
    send_command(direction)

    # ===== Draw Iris =====
    for idx in LEFT_IRIS + RIGHT_IRIS:
        x = int(face.landmark[idx].x * w)
        y = int(face.landmark[idx].y * h)
        cv2.circle(frame, (x, y), 3, (0, 255, 0), -1)

    for p in left_eye + right_eye:
        cv2.circle(frame, p, 1, (255, 0, 0), -1)

    # ===== Side Panel =====
    panel_w = 280
    panel   = np.ones((h, panel_w, 3), dtype=np.uint8) * 30

    colors = {
        "FORWARD":  (0, 180, 0),
        "LEFT":     (0, 140, 255),
        "RIGHT":    (0, 140, 255),
        "STOP":     (0, 0, 220),
        "BLINKING": (180, 0, 180),
    }
    box_color = colors.get(direction, (100, 100, 100))

    # Control status
    ctrl_color = (0, 200, 0) if not is_stopped else (0, 0, 220)
    ctrl_text  = "CONTROL: ON" if not is_stopped else "CONTROL: OFF"
    cv2.rectangle(panel, (10, 10), (panel_w-10, 60), ctrl_color, -1)
    cv2.putText(panel, ctrl_text, (15, 45),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    # Direction
    cv2.rectangle(panel, (10, 80), (panel_w-10, 150), box_color, -1)
    cv2.putText(panel, direction, (15, 133),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3)

    # Arrow
    cx, cy = panel_w // 2, 220
    if direction == "FORWARD":
        cv2.arrowedLine(panel, (cx, cy+40), (cx, cy-40), (0, 200, 0), 4, tipLength=0.4)
    elif direction == "LEFT":
        cv2.arrowedLine(panel, (cx+40, cy), (cx-40, cy), (0, 140, 255), 4, tipLength=0.4)
    elif direction == "RIGHT":
        cv2.arrowedLine(panel, (cx-40, cy), (cx+40, cy), (0, 140, 255), 4, tipLength=0.4)
    elif direction == "STOP":
        cv2.rectangle(panel, (cx-30, cy-30), (cx+30, cy+30), (0, 0, 220), -1)
        cv2.putText(panel, "||", (cx-18, cy+12),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3)
    elif direction == "BLINKING":
        cv2.putText(panel, "* *", (cx-30, cy+10),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (180, 0, 180), 3)

    # Stats
    bt_str = "BT: ON" if bt else "BT: OFF"
    bt_col = (0, 200, 0) if bt else (0, 0, 200)
    cv2.putText(panel, bt_str,              (10, h-100), cv2.FONT_HERSHEY_SIMPLEX, 0.6, bt_col, 2)
    cv2.putText(panel, f"FPS:   {fps:.1f}", (10, h-75),  cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
    cv2.putText(panel, f"EAR:   {ear:.2f}", (10, h-50),  cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 1)
    cv2.putText(panel, f"ratio: {ratio:.2f}",(10, h-25),  cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 1)

    # ===== Merge =====
    combined = np.hstack((frame, panel))
    cv2.imshow("EYEonic", combined)

    if cv2.waitKey(1) & 0xFF == 27:
        break

# ===== Cleanup =====
cap.release()
cv2.destroyAllWindows()
if bt:
    bt.close()
    print("🔚 Bluetooth closed.")

❌ BT Error: could not open port 'COM7': FileNotFoundError(2, 'The system cannot find the file specified.', None, 2)


In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import serial
import serial.tools.list_ports
from collections import deque
import time

# ===== Bluetooth Setup =====
def connect_bluetooth():
    ports = serial.tools.list_ports.comports()
    print(" Available ports:")
    for p in ports:
        print(f"   {p.device} - {p.description}")
    
    bt_port = None
    for p in ports:
        if any(x in p.description.upper() for x in ["BLUETOOTH", "HC-05", "HC-06", "ESP32", "COM"]):
            bt_port = p.device
            break
    
    if bt_port is None:
        print(" No Bluetooth found — running without Bluetooth")
        return None
    
    try:
        bt = serial.Serial(bt_port, baudrate=9600, timeout=1)
        print(f" Connected to {bt_port}")
        return bt
    except Exception as e:
        print(f" Bluetooth error: {e}")
        return None

bt = connect_bluetooth()

# ===== MediaPipe =====
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(refine_landmarks=True, max_num_faces=1)
cap = cv2.VideoCapture(0)

# ===== Landmarks =====
LEFT_IRIS   = [474, 475, 476, 477]
RIGHT_IRIS  = [469, 470, 471, 472]
LEFT_EYE_H  = [33, 133]
RIGHT_EYE_H = [362, 263]
LEFT_EYE    = [33, 160, 158, 133, 153, 144]
RIGHT_EYE   = [362, 385, 387, 263, 373, 380]

# ===== Blink Settings =====
BLINK_THRESHOLD  = 0.20
BLINK_FRAMES     = 3
blink_counter    = 0
blink_cooldown   = 0

# ===== Control State =====
# ===== Control State =====
is_stopped = False
close_counter = 0

# مدة غلق العين المطلوبة لتفعيل Start/Stop
CLOSE_FRAMES = 60

# Emergency
eye_closed_start = None
EMERGENCY_FRAMES = 450
emergency_sent = False
# ===== Smoothing =====
HISTORY_SIZE     = 10
direction_history = deque(maxlen=HISTORY_SIZE)

# ===== Bluetooth =====
last_command     = None
COMMAND_MAP      = {
    "FORWARD":  "F",
    "LEFT":     "L",
    "RIGHT":    "R",
    "STOP":     "S",
    "BLINKING": "B",         
}

# ===== FPS =====
frame_count  = 0
start_time   = time.time()
fps          = 0

def eye_aspect_ratio(eye):
    A = np.linalg.norm(np.array(eye[1]) - np.array(eye[5]))
    B = np.linalg.norm(np.array(eye[2]) - np.array(eye[4]))
    C = np.linalg.norm(np.array(eye[0]) - np.array(eye[3]))
    return (A + B) / (2.0 * C)

def get_iris_ratio(face, iris_ids, corner_ids):
    iris_x  = np.mean([face.landmark[i].x for i in iris_ids])
    x1 = face.landmark[corner_ids[0]].x
    x2 = face.landmark[corner_ids[1]].x
    left_x  = min(x1, x2)
    right_x = max(x1, x2)
    eye_w   = max(right_x - left_x, 0.001)
    return (iris_x - left_x) / eye_w

def send_command(direction):
    global last_command
    cmd = COMMAND_MAP.get(direction)
    if cmd and cmd != last_command and bt:
        try:
            bt.write(cmd.encode())
            last_command = cmd
            print(f" Sent: {cmd} ({direction})")
        except Exception as e:
            print(f" BT Error: {e}")

def smooth_direction(history):
    if not history:
        return "FORWARD"
    counts = {}
    for d in history:
        counts[d] = counts.get(d, 0) + 1
    return max(counts, key=counts.get)

# ===== Main Loop =====
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb)
    h, w, _ = frame.shape

    # ===== FPS =====
    frame_count += 1
    elapsed = time.time() - start_time
    if elapsed >= 1.0:
        fps = frame_count / elapsed
        frame_count = 0
        start_time = time.time()

    direction = "FORWARD"

    # ===== No Face =====
    if not results.multi_face_landmarks:
        cv2.rectangle(frame, (0, 0), (w, h), (0, 0, 80), 4)
        cv2.rectangle(frame, (10, 10), (420, 70), (0, 0, 200), -1)
        cv2.putText(frame, "NO FACE DETECTED", (20, 55),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.1, (255, 255, 255), 3)
        send_command("STOP")
        cv2.imshow("EYEonic", frame)
        if cv2.waitKey(1) & 0xFF == 27:
            break
        continue

    face = results.multi_face_landmarks[0]

    # ===== Blink Cooldown =====
    if blink_cooldown > 0:
        blink_cooldown -= 1

    # ===== Eye Points =====
    left_eye, right_eye = [], []
    for idx in LEFT_EYE:
        x = int(face.landmark[idx].x * w)
        y = int(face.landmark[idx].y * h)
        left_eye.append((x, y))
    for idx in RIGHT_EYE:
        x = int(face.landmark[idx].x * w)
        y = int(face.landmark[idx].y * h)
        right_eye.append((x, y))

    ear = (eye_aspect_ratio(left_eye) + eye_aspect_ratio(right_eye)) / 2

    # ===== Toggle STOP=====
    if ear < BLINK_THRESHOLD:
        close_counter += 1
        blink_counter += 1
    else:
        close_counter = 0
        if blink_counter >= BLINK_FRAMES and blink_cooldown == 0:
            pass  
        blink_counter = 0

    if close_counter >= CLOSE_FRAMES and blink_cooldown == 0:
        is_stopped = not is_stopped
        blink_cooldown = 30
        close_counter  = 0
        print(f"Control: {'STOP' if is_stopped else 'GO'}")

    # ===== Iris Ratio =====
    ratio = (get_iris_ratio(face, LEFT_IRIS, LEFT_EYE_H) +
             get_iris_ratio(face, RIGHT_IRIS, RIGHT_EYE_H)) / 2

    # ===== Raw Direction =====
    if is_stopped:
        raw_dir = "STOP"
    elif ear < BLINK_THRESHOLD:
        raw_dir = "BLINKING"
    elif ratio < 0.40:
        raw_dir = "LEFT"
    elif ratio > 0.60:
        raw_dir = "RIGHT"
    else:
        raw_dir = "FORWARD"

    # ===== Smoothing =====
    direction_history.append(raw_dir)
    direction = smooth_direction(direction_history)

    # ===== Send Bluetooth =====
    send_command(direction)

    # ===== Draw Iris =====
    for idx in LEFT_IRIS + RIGHT_IRIS:
        x = int(face.landmark[idx].x * w)
        y = int(face.landmark[idx].y * h)
        cv2.circle(frame, (x, y), 3, (0, 255, 0), -1)

    for p in left_eye + right_eye:
        cv2.circle(frame, p, 1, (255, 0, 0), -1)

    # ===== Side Panel =====
    panel_w = 280
    panel = np.ones((h, panel_w, 3), dtype=np.uint8) * 30

    colors = {
        "FORWARD":  (0, 180, 0),
        "LEFT":     (0, 140, 255),
        "RIGHT":    (0, 140, 255),
        "STOP":     (0, 0, 220),
        "BLINKING": (180, 0, 180),
    }
    box_color = colors.get(direction, (100, 100, 100))

    # Control status
    ctrl_color = (0, 200, 0) if not is_stopped else (0, 0, 220)
    ctrl_text  = "CONTROL: ON" if not is_stopped else "CONTROL: OFF"
    cv2.rectangle(panel, (10, 10), (panel_w-10, 60), ctrl_color, -1)
    cv2.putText(panel, ctrl_text, (15, 45),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)

    # Direction
    cv2.rectangle(panel, (10, 80), (panel_w-10, 150), box_color, -1)
    cv2.putText(panel, direction, (15, 133),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255,255,255), 3)

    # Arrow
    cx, cy = panel_w // 2, 210
    if direction == "FORWARD":
        cv2.arrowedLine(panel, (cx, cy+40), (cx, cy-40), (0,200,0), 4, tipLength=0.4)
    elif direction == "LEFT":
        cv2.arrowedLine(panel, (cx+40, cy), (cx-40, cy), (0,140,255), 4, tipLength=0.4)
    elif direction == "RIGHT":
        cv2.arrowedLine(panel, (cx-40, cy), (cx+40, cy), (0,140,255), 4, tipLength=0.4)
    elif direction == "STOP":
        cv2.rectangle(panel, (cx-30, cy-30), (cx+30, cy+30), (0,0,220), -1)
        cv2.putText(panel, "||", (cx-18, cy+12),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255,255,255), 3)
    elif direction == "BLINKING":
        cv2.putText(panel, "* *", (cx-30, cy+10),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (180,0,180), 3)

    # Stats
    bt_status = " BT: ON" if bt else " BT: OFF"
    cv2.putText(panel, bt_status,        (10, h-100), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 1)
    cv2.putText(panel, f"FPS: {fps:.1f}", (10, h-70),  cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 1)
    cv2.putText(panel, f"EAR: {ear:.2f}", (10, h-45),  cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,0),   1)
    cv2.putText(panel, f"ratio:{ratio:.2f}",(10, h-20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,0),   1)

    # ===== Merge =====
    combined = np.hstack((frame, panel))
    cv2.imshow("EYEonic", combined)
    

    if cv2.waitKey(1) & 0xFF == 27:
        break

# ===== Cleanup =====
cap.release()
cv2.destroyAllWindows()
if bt:
    bt.close()
    print(" Bluetooth closed.")

 Available ports:
   COM3 - Silicon Labs CP210x USB to UART Bridge (COM3)
   COM5 - Standard Serial over Bluetooth link (COM5)
   COM6 - Standard Serial over Bluetooth link (COM6)
 Bluetooth error: could not open port 'COM3': PermissionError(13, 'Access is denied.', None, 5)


In [5]:
import cv2
import mediapipe as mp
import numpy as np
from collections import deque
import time

# ===== Wifi Setup =====
import socket

ESP32_IP = "192.168.4.1"
ESP32_PORT = 5005

sock = socket.socket(
    socket.AF_INET,
    socket.SOCK_DGRAM
)
# ===== MediaPipe =====
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(refine_landmarks=True, max_num_faces=1)
cap = cv2.VideoCapture(0)

# ===== Landmarks =====
LEFT_IRIS   = [474, 475, 476, 477]
RIGHT_IRIS  = [469, 470, 471, 472]
LEFT_EYE_H  = [33, 133]
RIGHT_EYE_H = [362, 263]
LEFT_EYE    = [33, 160, 158, 133, 153, 144]
RIGHT_EYE   = [362, 385, 387, 263, 373, 380]

# ===== Blink Settings =====
BLINK_THRESHOLD  = 0.20
BLINK_FRAMES     = 3
blink_counter    = 0
blink_cooldown   = 0

# ===== Control State =====
is_stopped       = False
close_counter    = 0
eye_closed_start = None      
EMERGENCY_FRAMES = 450   # 15 ثانية تقريباً عند 30 FPS
emergency_sent = False
# ===== Smoothing =====
HISTORY_SIZE     = 10
direction_history = deque(maxlen=HISTORY_SIZE)

# ===== Wifi =====
last_command     = None
COMMAND_MAP      = {
    "FORWARD":  "F",
    "LEFT":     "L",
    "RIGHT":    "R",
    "STOP":     "S",
    "BLINKING": "S",
    "EMERGENCY": "E",
}

# ===== FPS =====
frame_count  = 0
start_time   = time.time()
fps          = 0

def eye_aspect_ratio(eye):
    A = np.linalg.norm(np.array(eye[1]) - np.array(eye[5]))
    B = np.linalg.norm(np.array(eye[2]) - np.array(eye[4]))
    C = np.linalg.norm(np.array(eye[0]) - np.array(eye[3]))
    return (A + B) / (2.0 * C)

def get_iris_ratio(face, iris_ids, corner_ids):
    iris_x  = np.mean([face.landmark[i].x for i in iris_ids])
    x1 = face.landmark[corner_ids[0]].x
    x2 = face.landmark[corner_ids[1]].x
    left_x  = min(x1, x2)
    right_x = max(x1, x2)
    eye_w   = max(right_x - left_x, 0.001)
    return (iris_x - left_x) / eye_w

def send_command(direction):

    global last_command

    cmd = COMMAND_MAP.get(direction)

    if cmd and cmd != last_command:

        try:

            sock.sendto(
                cmd.encode(),
                (ESP32_IP, ESP32_PORT)
            )

            last_command = cmd

            print(f"Sent: {cmd}")

        except Exception as e:

            print("UDP Error:", e)
def smooth_direction(history):
    if not history:
        return "FORWARD"
    counts = {}
    for d in history:
        counts[d] = counts.get(d, 0) + 1
    return max(counts, key=counts.get)

# ===== Main Loop =====
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb)
    h, w, _ = frame.shape

    # ===== FPS =====
    frame_count += 1
    elapsed = time.time() - start_time
    if elapsed >= 1.0:
        fps = frame_count / elapsed
        frame_count = 0
        start_time = time.time()

    direction = "FORWARD"

    # ===== No Face =====
    if not results.multi_face_landmarks:
        cv2.rectangle(frame, (0, 0), (w, h), (0, 0, 80), 4)
        cv2.rectangle(frame, (10, 10), (420, 70), (0, 0, 200), -1)
        cv2.putText(frame, "NO FACE DETECTED", (20, 55),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.1, (255, 255, 255), 3)
        send_command("STOP")
        cv2.imshow("EYEonic", frame)
        if cv2.waitKey(1) & 0xFF == 27:
            break
        continue

    face = results.multi_face_landmarks[0]

    # ===== Blink Cooldown =====
    if blink_cooldown > 0:
        blink_cooldown -= 1

    # ===== Eye Points =====
    left_eye, right_eye = [], []
    for idx in LEFT_EYE:
        x = int(face.landmark[idx].x * w)
        y = int(face.landmark[idx].y * h)
        left_eye.append((x, y))
    for idx in RIGHT_EYE:
        x = int(face.landmark[idx].x * w)
        y = int(face.landmark[idx].y * h)
        right_eye.append((x, y))

    ear = (eye_aspect_ratio(left_eye) + eye_aspect_ratio(right_eye)) / 2
# ===== Emergency Timer =====

if ear < BLINK_THRESHOLD:

    if eye_closed_start is None:
        eye_closed_start = time.time()

    elif time.time() - eye_closed_start >= 15 and not emergency_sent:

        print("EMERGENCY MODE ACTIVATED")

        send_command("EMERGENCY")

        emergency_sent = True

else:

    eye_closed_start = None
    emergency_sent = False
    # ===== Toggle STOP=====
    if ear < BLINK_THRESHOLD:
        close_counter += 1
        blink_counter += 1
    else:
        close_counter = 0
        if blink_counter >= BLINK_FRAMES and blink_cooldown == 0:
            pass  
        blink_counter = 0

    if close_counter >= CLOSE_FRAMES and blink_cooldown == 0:
        is_stopped = not is_stopped
        blink_cooldown = 30
        close_counter  = 0
        print(f"Control: {'STOP' if is_stopped else 'GO'}")
        # ===== Emergency Mode =====
if close_counter >= EMERGENCY_FRAMES and not emergency_sent:

    print("EMERGENCY MODE ACTIVATED")

    send_command("EMERGENCY")

    emergency_sent = True

if close_counter == 0:
    emergency_sent = False
   
    # ===== Iris Ratio =====
    ratio = (get_iris_ratio(face, LEFT_IRIS, LEFT_EYE_H) +
             get_iris_ratio(face, RIGHT_IRIS, RIGHT_EYE_H)) / 2

    # ===== Raw Direction =====
    if is_stopped:
        raw_dir = "STOP"
    elif ear < BLINK_THRESHOLD:
        raw_dir = "BLINKING"
    elif ratio < 0.40:
        raw_dir = "LEFT"
    elif ratio > 0.60:
        raw_dir = "RIGHT"
    else:
        raw_dir = "FORWARD"

    # ===== Smoothing =====
    direction_history.append(raw_dir)
    direction = smooth_direction(direction_history)

    # ===== Send Wifi =====
    send_command(direction)

    # ===== Draw Iris =====
    for idx in LEFT_IRIS + RIGHT_IRIS:
        x = int(face.landmark[idx].x * w)
        y = int(face.landmark[idx].y * h)
        cv2.circle(frame, (x, y), 3, (0, 255, 0), -1)

    for p in left_eye + right_eye:
        cv2.circle(frame, p, 1, (255, 0, 0), -1)

    # ===== Side Panel =====
    panel_w = 280
    panel = np.ones((h, panel_w, 3), dtype=np.uint8) * 30

    colors = {
        "FORWARD":  (0, 180, 0),
        "LEFT":     (0, 140, 255),
        "RIGHT":    (0, 140, 255),
        "STOP":     (0, 0, 220),
        "BLINKING": (180, 0, 180),
    }
    box_color = colors.get(direction, (100, 100, 100))

    # Control status
    ctrl_color = (0, 200, 0) if not is_stopped else (0, 0, 220)
    ctrl_text  = "CONTROL: ON" if not is_stopped else "CONTROL: OFF"
    cv2.rectangle(panel, (10, 10), (panel_w-10, 60), ctrl_color, -1)
    cv2.putText(panel, ctrl_text, (15, 45),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)

    # Direction
    cv2.rectangle(panel, (10, 80), (panel_w-10, 150), box_color, -1)
    cv2.putText(panel, direction, (15, 133),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255,255,255), 3)

    # Arrow
    cx, cy = panel_w // 2, 210
    if direction == "FORWARD":
        cv2.arrowedLine(panel, (cx, cy+40), (cx, cy-40), (0,200,0), 4, tipLength=0.4)
    elif direction == "LEFT":
        cv2.arrowedLine(panel, (cx+40, cy), (cx-40, cy), (0,140,255), 4, tipLength=0.4)
    elif direction == "RIGHT":
        cv2.arrowedLine(panel, (cx-40, cy), (cx+40, cy), (0,140,255), 4, tipLength=0.4)
    elif direction == "STOP":
        cv2.rectangle(panel, (cx-30, cy-30), (cx+30, cy+30), (0,0,220), -1)
        cv2.putText(panel, "||", (cx-18, cy+12),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255,255,255), 3)
    elif direction == "BLINKING":
        cv2.putText(panel, "* *", (cx-30, cy+10),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (180,0,180), 3)

    # Stats
    bt_status = " WIFI: ON"
    cv2.putText(panel, bt_status,        (10, h-100), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 1)
    cv2.putText(panel, f"FPS: {fps:.1f}", (10, h-70),  cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 1)
    cv2.putText(panel, f"EAR: {ear:.2f}", (10, h-45),  cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,0),   1)
    cv2.putText(panel, f"ratio:{ratio:.2f}",(10, h-20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,0),   1)

    # ===== Merge =====
    combined = np.hstack((frame, panel))
    cv2.imshow("EYEonic", combined)
    

    if cv2.waitKey(1) & 0xFF == 27:
        break

# ===== Cleanup =====
cap.release()
cv2.destroyAllWindows()

Sent: S


NameError: name 'CLOSE_FRAMES' is not defined

In [1]:
import cv2
import mediapipe as mp
import numpy as np
from collections import deque
import time
import socket

# ===== WiFi Setup =====
ESP32_IP   = "192.168.4.1"
ESP32_PORT = 5005
sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

# ===== MediaPipe =====
mp_face_mesh = mp.solutions.face_mesh
face_mesh    = mp_face_mesh.FaceMesh(refine_landmarks=True, max_num_faces=1)
cap          = cv2.VideoCapture(0)
print("📷 Camera ready:", cap.isOpened())

# ===== Landmarks =====
LEFT_IRIS   = [474, 475, 476, 477]
RIGHT_IRIS  = [469, 470, 471, 472]
LEFT_EYE_H  = [33, 133]
RIGHT_EYE_H = [362, 263]
LEFT_EYE    = [33, 160, 158, 133, 153, 144]
RIGHT_EYE   = [362, 385, 387, 263, 373, 380]

# ===== Blink Settings =====
BLINK_THRESHOLD = 0.20
blink_cooldown  = 0
close_counter   = 0

CLOSE_FRAMES_STOP  = 60    # ~2 ثانية  → toggle STOP/GO
CLOSE_FRAMES_EMERG = 450   # ~15 ثانية → بعت E

# ===== Control State =====
is_stopped        = False
emergency_sent    = False

# ===== Smoothing =====
direction_history = deque(maxlen=10)

# ===== Commands =====
last_command = None
COMMAND_MAP  = {
    "FORWARD":   "F",
    "LEFT":      "L",
    "RIGHT":     "R",
    "STOP":      "S",
    "BLINKING":  "S",
    "EMERGENCY": "E",
}

# ===== FPS =====
frame_count = 0
start_time  = time.time()
fps         = 0

# ===== Functions =====
def eye_aspect_ratio(eye):
    A = np.linalg.norm(np.array(eye[1]) - np.array(eye[5]))
    B = np.linalg.norm(np.array(eye[2]) - np.array(eye[4]))
    C = np.linalg.norm(np.array(eye[0]) - np.array(eye[3]))
    return (A + B) / (2.0 * C)

def get_iris_ratio(face, iris_ids, corner_ids):
    iris_x  = np.mean([face.landmark[i].x for i in iris_ids])
    x1      = face.landmark[corner_ids[0]].x
    x2      = face.landmark[corner_ids[1]].x
    left_x  = min(x1, x2)
    right_x = max(x1, x2)
    eye_w   = max(right_x - left_x, 0.001)
    return (iris_x - left_x) / eye_w

def send_command(direction):
    global last_command
    cmd = COMMAND_MAP.get(direction)
    if cmd and cmd != last_command:
        try:
            sock.sendto(cmd.encode(), (ESP32_IP, ESP32_PORT))
            last_command = cmd
            print(f"📡 Sent: {cmd} ({direction})")
        except Exception as e:
            print(f"❌ UDP Error: {e}")

def smooth_direction(history):
    if not history:
        return "FORWARD"
    counts = {}
    for d in history:
        counts[d] = counts.get(d, 0) + 1
    return max(counts, key=counts.get)

# ===== Main Loop =====
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb)
    h, w, _ = frame.shape

    # FPS
    frame_count += 1
    elapsed = time.time() - start_time
    if elapsed >= 1.0:
        fps         = frame_count / elapsed
        frame_count = 0
        start_time  = time.time()

    direction = "FORWARD"

    # ===== No Face =====
    if not results.multi_face_landmarks:
        cv2.rectangle(frame, (0, 0), (w, h), (0, 0, 80), 4)
        cv2.rectangle(frame, (10, 10), (420, 70), (0, 0, 200), -1)
        cv2.putText(frame, "NO FACE DETECTED", (20, 55),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.1, (255, 255, 255), 3)
        send_command("STOP")
        cv2.imshow("EYEonic", frame)
        if cv2.waitKey(1) & 0xFF == 27:
            break
        continue

    face = results.multi_face_landmarks[0]

    if blink_cooldown > 0:
        blink_cooldown -= 1

    # ===== Eye Points =====
    left_eye, right_eye = [], []
    for idx in LEFT_EYE:
        x = int(face.landmark[idx].x * w)
        y = int(face.landmark[idx].y * h)
        left_eye.append((x, y))
    for idx in RIGHT_EYE:
        x = int(face.landmark[idx].x * w)
        y = int(face.landmark[idx].y * h)
        right_eye.append((x, y))

    ear = (eye_aspect_ratio(left_eye) + eye_aspect_ratio(right_eye)) / 2

    # ===== Eye Close Counter =====
    if ear < BLINK_THRESHOLD:
        close_counter += 1
    else:
        # ===== لما العين تتفتح — reset =====
        close_counter  = 0
        emergency_sent = False

    # ===== 15 ثانية → Emergency =====
    if close_counter >= CLOSE_FRAMES_EMERG and not emergency_sent:
        send_command("EMERGENCY")
        emergency_sent = True
        print("🚨 EMERGENCY SENT")

    # ===== 2 ثانية → Toggle STOP/GO =====
    elif close_counter >= CLOSE_FRAMES_STOP and blink_cooldown == 0 and not emergency_sent:
        is_stopped     = not is_stopped
        blink_cooldown = 40
        close_counter  = CLOSE_FRAMES_STOP  # مش بيريسيت عشان يكمل عد لـ 15 ثانية
        print(f"🚦 Control: {'STOP' if is_stopped else 'GO'}")

    # ===== Iris Ratio =====
    ratio = (get_iris_ratio(face, LEFT_IRIS, LEFT_EYE_H) +
             get_iris_ratio(face, RIGHT_IRIS, RIGHT_EYE_H)) / 2

    # ===== Direction =====
    if emergency_sent:
        raw_dir = "EMERGENCY"
    elif is_stopped:
        raw_dir = "STOP"
    elif ear < BLINK_THRESHOLD:
        raw_dir = "BLINKING"
    elif ratio < 0.40:
        raw_dir = "LEFT"
    elif ratio > 0.60:
        raw_dir = "RIGHT"
    else:
        raw_dir = "FORWARD"

    direction_history.append(raw_dir)
    direction = smooth_direction(direction_history)
    send_command(direction)

    # ===== Draw Iris =====
    for idx in LEFT_IRIS + RIGHT_IRIS:
        x = int(face.landmark[idx].x * w)
        y = int(face.landmark[idx].y * h)
        cv2.circle(frame, (x, y), 3, (0, 255, 0), -1)

    for p in left_eye + right_eye:
        cv2.circle(frame, p, 1, (255, 0, 0), -1)

    # ===== Side Panel =====
    panel_w = 280
    panel   = np.ones((h, panel_w, 3), dtype=np.uint8) * 30

    colors = {
        "FORWARD":   (0, 180, 0),
        "LEFT":      (0, 140, 255),
        "RIGHT":     (0, 140, 255),
        "STOP":      (0, 0, 220),
        "BLINKING":  (180, 0, 180),
        "EMERGENCY": (0, 0, 255),
    }
    box_color  = colors.get(direction, (100, 100, 100))
    ctrl_color = (0, 200, 0) if not is_stopped else (0, 0, 220)
    ctrl_text  = "CONTROL: ON" if not is_stopped else "CONTROL: OFF"

    cv2.rectangle(panel, (10, 10), (panel_w-10, 60), ctrl_color, -1)
    cv2.putText(panel, ctrl_text, (15, 45),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    cv2.rectangle(panel, (10, 80), (panel_w-10, 150), box_color, -1)
    cv2.putText(panel, direction, (15, 133),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 3)

    # Eye close progress bar
    bar_max = panel_w - 20
    bar_val = int(min(close_counter / CLOSE_FRAMES_EMERG, 1.0) * bar_max)
    bar_col = (0, 0, 255) if close_counter >= CLOSE_FRAMES_EMERG else (0, 200, 255)
    cv2.rectangle(panel, (10, 160), (panel_w-10, 180), (60, 60, 60), -1)
    cv2.rectangle(panel, (10, 160), (10 + bar_val, 180), bar_col, -1)
    secs = min(close_counter / 30, 15)
    cv2.putText(panel, f"Eye closed: {secs:.1f}s / 15s", (10, 195),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200, 200, 200), 1)

    # Arrow
    cx, cy = panel_w // 2, 230
    if direction == "FORWARD":
        cv2.arrowedLine(panel, (cx, cy+40), (cx, cy-40), (0, 200, 0), 4, tipLength=0.4)
    elif direction == "LEFT":
        cv2.arrowedLine(panel, (cx+40, cy), (cx-40, cy), (0, 140, 255), 4, tipLength=0.4)
    elif direction == "RIGHT":
        cv2.arrowedLine(panel, (cx-40, cy), (cx+40, cy), (0, 140, 255), 4, tipLength=0.4)
    elif direction == "STOP":
        cv2.rectangle(panel, (cx-30, cy-30), (cx+30, cy+30), (0, 0, 220), -1)
        cv2.putText(panel, "||", (cx-18, cy+12),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3)
    elif direction == "EMERGENCY":
        cv2.putText(panel, "SOS", (cx-30, cy+10),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)

    # Stats
    cv2.putText(panel, "WiFi: ON",          (10, h-100), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 0), 1)
    cv2.putText(panel, f"FPS:   {fps:.1f}", (10, h-75),  cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
    cv2.putText(panel, f"EAR:   {ear:.2f}", (10, h-50),  cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 1)
    cv2.putText(panel, f"ratio: {ratio:.2f}",(10, h-25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 1)

    combined = np.hstack((frame, panel))
    cv2.imshow("EYEonic", combined)

    if cv2.waitKey(1) & 0xFF == 27:
        break

# ===== Cleanup =====
cap.release()
cv2.destroyAllWindows()
sock.close()
print("🔚 WiFi closed.")

📷 Camera ready: True
📡 Sent: F (FORWARD)
📡 Sent: R (RIGHT)
📡 Sent: F (FORWARD)
📡 Sent: R (RIGHT)
📡 Sent: F (FORWARD)
📡 Sent: R (RIGHT)
📡 Sent: F (FORWARD)
📡 Sent: R (RIGHT)
📡 Sent: F (FORWARD)
📡 Sent: R (RIGHT)
📡 Sent: F (FORWARD)
📡 Sent: R (RIGHT)
📡 Sent: F (FORWARD)
📡 Sent: R (RIGHT)
📡 Sent: F (FORWARD)
📡 Sent: R (RIGHT)
📡 Sent: F (FORWARD)
📡 Sent: R (RIGHT)
📡 Sent: F (FORWARD)
📡 Sent: R (RIGHT)
📡 Sent: S (BLINKING)
🚦 Control: STOP
🚦 Control: GO
🚦 Control: STOP
🚦 Control: GO
📡 Sent: F (FORWARD)
📡 Sent: S (BLINKING)
📡 Sent: F (FORWARD)
📡 Sent: S (BLINKING)
🚦 Control: STOP
🚦 Control: GO
📡 Sent: R (RIGHT)
📡 Sent: F (FORWARD)
📡 Sent: R (RIGHT)
📡 Sent: L (LEFT)
📡 Sent: F (FORWARD)
📡 Sent: R (RIGHT)
📡 Sent: F (FORWARD)
📡 Sent: R (RIGHT)
📡 Sent: S (BLINKING)
📡 Sent: R (RIGHT)
📡 Sent: S (BLINKING)
📡 Sent: F (FORWARD)
📡 Sent: S (STOP)
📡 Sent: F (FORWARD)
📡 Sent: S (STOP)
📡 Sent: F (FORWARD)
📡 Sent: S (STOP)
📡 Sent: F (FORWARD)
📡 Sent: S (BLINKING)
📡 Sent: R (RIGHT)
📡 Sent: S (BLINKING)
📡 Sent

In [10]:
import cv2
import mediapipe as mp
import numpy as np
from collections import deque
import time
import socket

# ===================================================
#   EYEonic — Gaze-Controlled Robotic Car
#   Real-time Eye Tracking & Wireless Control System
# ===================================================

# ===== WiFi Setup =====
ESP32_IP   = "192.168.4.1"
ESP32_PORT = 5005
sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

# ===== MediaPipe =====
mp_face_mesh = mp.solutions.face_mesh
face_mesh    = mp_face_mesh.FaceMesh(refine_landmarks=True, max_num_faces=1)
cap          = cv2.VideoCapture(0)
print("📷 Camera ready:", cap.isOpened())

# ===== Landmarks =====
LEFT_IRIS   = [474, 475, 476, 477]
RIGHT_IRIS  = [469, 470, 471, 472]
LEFT_EYE_H  = [33, 133]
RIGHT_EYE_H = [362, 263]
LEFT_EYE    = [33, 160, 158, 133, 153, 144]
RIGHT_EYE   = [362, 385, 387, 263, 373, 380]

# ===== Thresholds =====
BLINK_THRESHOLD    = 0.20
FPS_ESTIMATE       = 30
CLOSE_SEC_STOP     = 2
CLOSE_SEC_EMERG    = 15
CLOSE_FRAMES_STOP  = FPS_ESTIMATE * CLOSE_SEC_STOP
CLOSE_FRAMES_EMERG = FPS_ESTIMATE * CLOSE_SEC_EMERG

# ===== State =====
is_stopped      = False
close_counter   = 0
toggle_done     = False     # ✅ مرة واحدة بس عند 2 ثانية
emergency_sent  = False     # ✅ مرة واحدة بس عند 15 ثانية

# ===== Smoothing =====
direction_history = deque(maxlen=10)

# ===== Commands =====
last_command = None
COMMAND_MAP  = {
    "FORWARD":   "F",
    "LEFT":      "L",
    "RIGHT":     "R",
    "STOP":      "S",
    "BLINKING":  "S",
    "EMERGENCY": "E",
}

# ===== FPS =====
frame_count = 0
start_time  = time.time()
fps         = 0

# ===== Colors (BGR) =====
COL_GREEN   = (0,   200,   0)
COL_BLUE    = (255, 140,   0)
COL_RED     = (0,     0, 220)
COL_PURPLE  = (180,   0, 180)
COL_ORANGE  = (0,   165, 255)
COL_DARK    = (30,   30,  30)
COL_WHITE   = (255, 255, 255)
COL_YELLOW  = (0,   255, 255)

# ==================== Functions ====================

def eye_aspect_ratio(eye):
    A = np.linalg.norm(np.array(eye[1]) - np.array(eye[5]))
    B = np.linalg.norm(np.array(eye[2]) - np.array(eye[4]))
    C = np.linalg.norm(np.array(eye[0]) - np.array(eye[3]))
    return (A + B) / (2.0 * C)

def get_iris_ratio(face, iris_ids, corner_ids):
    iris_x  = np.mean([face.landmark[i].x for i in iris_ids])
    x1      = face.landmark[corner_ids[0]].x
    x2      = face.landmark[corner_ids[1]].x
    left_x  = min(x1, x2)
    right_x = max(x1, x2)
    return (iris_x - left_x) / max(right_x - left_x, 0.001)

def send_command(direction):
    global last_command
    cmd = COMMAND_MAP.get(direction)
    if cmd and cmd != last_command:
        try:
            sock.sendto(cmd.encode(), (ESP32_IP, ESP32_PORT))
            last_command = cmd
            print(f"📡 Sent: {cmd} ({direction})")
        except Exception as e:
            print(f"❌ UDP Error: {e}")

def smooth_direction(history):
    if not history:
        return "FORWARD"
    counts = {}
    for d in history:
        counts[d] = counts.get(d, 0) + 1
    return max(counts, key=counts.get)

def draw_panel(panel, direction, ear, ratio, fps,
               close_counter, is_stopped, emergency_sent):
    h, panel_w = panel.shape[:2]

    COLORS = {
        "FORWARD":   COL_GREEN,
        "LEFT":      COL_BLUE,
        "RIGHT":     COL_BLUE,
        "STOP":      COL_RED,
        "BLINKING":  COL_PURPLE,
        "EMERGENCY": (0, 0, 255),
    }

    # ── Header ──
    cv2.rectangle(panel, (0, 0), (panel_w, 50), (20, 20, 20), -1)
    cv2.putText(panel, "EYEonic", (15, 35),
                cv2.FONT_HERSHEY_DUPLEX, 1.0, COL_YELLOW, 2)

    # ── Control Status ──
    ctrl_col  = COL_GREEN if not is_stopped else COL_RED
    ctrl_text = "ACTIVE" if not is_stopped else "PAUSED"
    cv2.rectangle(panel, (10, 60), (panel_w-10, 105), ctrl_col, -1)
    cv2.rectangle(panel, (10, 60), (panel_w-10, 105), COL_WHITE, 1)
    cv2.putText(panel, ctrl_text, (20, 93),
                cv2.FONT_HERSHEY_DUPLEX, 1.0, COL_WHITE, 2)

    # ── Direction Box ──
    box_col = COLORS.get(direction, (80, 80, 80))
    cv2.rectangle(panel, (10, 115), (panel_w-10, 175), box_col, -1)
    cv2.rectangle(panel, (10, 115), (panel_w-10, 175), COL_WHITE, 1)
    cv2.putText(panel, direction, (15, 162),
                cv2.FONT_HERSHEY_DUPLEX, 0.9, COL_WHITE, 2)

    # ── Arrow ──
    cx, cy = panel_w // 2, 220
    if direction == "FORWARD":
        cv2.arrowedLine(panel, (cx, cy+40), (cx, cy-40),
                        COL_GREEN, 4, tipLength=0.4)
    elif direction == "LEFT":
        cv2.arrowedLine(panel, (cx+40, cy), (cx-40, cy),
                        COL_BLUE, 4, tipLength=0.4)
    elif direction == "RIGHT":
        cv2.arrowedLine(panel, (cx-40, cy), (cx+40, cy),
                        COL_BLUE, 4, tipLength=0.4)
    elif direction == "STOP":
        cv2.rectangle(panel, (cx-28, cy-28), (cx+28, cy+28), COL_RED, -1)
        cv2.putText(panel, "||", (cx-16, cy+12),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, COL_WHITE, 3)
    elif direction == "EMERGENCY":
        cv2.putText(panel, "SOS", (cx-36, cy+14),
                    cv2.FONT_HERSHEY_DUPLEX, 1.3, (0, 0, 255), 3)
    elif direction == "BLINKING":
        cv2.putText(panel, "- -", (cx-28, cy+12),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, COL_PURPLE, 3)

    # ── Eye Close Progress Bar ──
    bar_y    = 270
    bar_w    = panel_w - 20
    progress = min(close_counter / CLOSE_FRAMES_EMERG, 1.0)
    secs     = min(close_counter / FPS_ESTIMATE, CLOSE_SEC_EMERG)

    bar_col = (0, 0, 255) if emergency_sent else \
              COL_ORANGE  if secs >= CLOSE_SEC_STOP else \
              (0, 200, 255)

    cv2.rectangle(panel, (10, bar_y), (panel_w-10, bar_y+18), (60, 60, 60), -1)
    cv2.rectangle(panel, (10, bar_y), (10 + int(bar_w * progress), bar_y+18),
                  bar_col, -1)
    cv2.rectangle(panel, (10, bar_y), (panel_w-10, bar_y+18), COL_WHITE, 1)

    # خط عند 2 ثانية
    mark_x = 10 + int(bar_w * (CLOSE_SEC_STOP / CLOSE_SEC_EMERG))
    cv2.line(panel, (mark_x, bar_y), (mark_x, bar_y+18), COL_WHITE, 2)

    cv2.putText(panel, f"Eye Closed: {secs:.1f}s / {CLOSE_SEC_EMERG}s",
                (10, bar_y + 34),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200, 200, 200), 1)
    cv2.putText(panel, f"2s=Toggle  15s=Emergency",
                (10, bar_y + 50),
                cv2.FONT_HERSHEY_SIMPLEX, 0.40, (150, 150, 150), 1)

    # ── Stats ──
    stats = [
        ("WiFi",  "ON",        (0, 200, 0)),
        ("FPS",   f"{fps:.0f}", (200, 200, 200)),
        ("EAR",   f"{ear:.2f}", COL_YELLOW),
        ("RATIO", f"{ratio:.2f}", COL_YELLOW),
    ]
    for i, (label, val, col) in enumerate(stats):
        y = h - 90 + i * 22
        cv2.putText(panel, f"{label}: {val}", (10, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, col, 1)

# ======================== Main Loop ========================

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb)
    h, w, _ = frame.shape

    # FPS
    frame_count += 1
    elapsed = time.time() - start_time
    if elapsed >= 1.0:
        fps         = frame_count / elapsed
        frame_count = 0
        start_time  = time.time()

    direction = "FORWARD"

    # ── No Face ──
    if not results.multi_face_landmarks:
        overlay = frame.copy()
        cv2.rectangle(overlay, (0, 0), (w, h), (0, 0, 60), -1)
        cv2.addWeighted(overlay, 0.3, frame, 0.7, 0, frame)
        cv2.rectangle(frame, (w//2-200, h//2-40), (w//2+200, h//2+40),
                      (0, 0, 180), -1)
        cv2.putText(frame, "NO FACE DETECTED", (w//2-185, h//2+12),
                    cv2.FONT_HERSHEY_DUPLEX, 0.9, COL_WHITE, 2)
        send_command("STOP")
        cv2.imshow("EYEonic — Gaze Control", frame)
        if cv2.waitKey(1) & 0xFF == 27:
            break
        continue

    face = results.multi_face_landmarks[0]

    # ── Eye Points ──
    left_eye, right_eye = [], []
    for idx in LEFT_EYE:
        left_eye.append((int(face.landmark[idx].x * w),
                         int(face.landmark[idx].y * h)))
    for idx in RIGHT_EYE:
        right_eye.append((int(face.landmark[idx].x * w),
                          int(face.landmark[idx].y * h)))

    ear = (eye_aspect_ratio(left_eye) + eye_aspect_ratio(right_eye)) / 2

    # ── Eye Close Logic ──
    if ear < BLINK_THRESHOLD:
        close_counter += 1

        # 15 ثانية → Emergency (مرة واحدة بس)
        if close_counter >= CLOSE_FRAMES_EMERG and not emergency_sent:
            emergency_sent = True
            send_command("EMERGENCY")
            print("🚨 EMERGENCY SENT")

        # 2 ثانية → Toggle (مرة واحدة بس — العداد بيكمل للـ 15)
        elif close_counter >= CLOSE_FRAMES_STOP and not toggle_done and not emergency_sent:
            toggle_done = True
            is_stopped  = not is_stopped
            print(f"🚦 Control: {'STOP' if is_stopped else 'GO'}")

    else:
        # العين اتفتحت — reset كل حاجة
        close_counter  = 0
        toggle_done    = False
        emergency_sent = False

    # ── Iris Ratio ──
    ratio = (get_iris_ratio(face, LEFT_IRIS,  LEFT_EYE_H) +
             get_iris_ratio(face, RIGHT_IRIS, RIGHT_EYE_H)) / 2

    # ── Direction ──
    if emergency_sent:
        raw_dir = "EMERGENCY"
    elif is_stopped:
        raw_dir = "STOP"
    elif ear < BLINK_THRESHOLD:
        raw_dir = "BLINKING"
    elif ratio < 0.40:
        raw_dir = "LEFT"
    elif ratio > 0.60:
        raw_dir = "RIGHT"
    else:
        raw_dir = "FORWARD"

    direction_history.append(raw_dir)
    direction = smooth_direction(direction_history)
    send_command(direction)

    # ── Draw Iris & Eye Points ──
    for idx in LEFT_IRIS + RIGHT_IRIS:
        cv2.circle(frame,
                   (int(face.landmark[idx].x * w),
                    int(face.landmark[idx].y * h)),
                   3, COL_GREEN, -1)
    for p in left_eye + right_eye:
        cv2.circle(frame, p, 1, COL_BLUE, -1)

    # ── Side Panel ──
    panel = np.full((h, 300, 3), COL_DARK, dtype=np.uint8)
    draw_panel(panel, direction, ear, ratio, fps,
               close_counter, is_stopped, emergency_sent)

    combined = np.hstack((frame, panel))
    cv2.imshow("EYEonic — Gaze Control", combined)

    if cv2.waitKey(1) & 0xFF == 27:
        break

# ── Cleanup ──
cap.release()
cv2.destroyAllWindows()
sock.close()
print("🔚 Session ended.")

📷 Camera ready: True
📡 Sent: L (LEFT)
📡 Sent: R (RIGHT)
📡 Sent: F (FORWARD)
📡 Sent: R (RIGHT)
📡 Sent: F (FORWARD)
📡 Sent: R (RIGHT)
📡 Sent: F (FORWARD)
📡 Sent: R (RIGHT)
📡 Sent: S (BLINKING)
🚦 Control: STOP
🚦 Control: GO
📡 Sent: R (RIGHT)
📡 Sent: F (FORWARD)
📡 Sent: R (RIGHT)
📡 Sent: F (FORWARD)
📡 Sent: S (BLINKING)
📡 Sent: R (RIGHT)
📡 Sent: S (BLINKING)
📡 Sent: F (FORWARD)
📡 Sent: R (RIGHT)
📡 Sent: F (FORWARD)
📡 Sent: S (BLINKING)
🚦 Control: STOP
📡 Sent: E (EMERGENCY)
🚨 EMERGENCY SENT
📡 Sent: S (STOP)
📡 Sent: E (EMERGENCY)
📡 Sent: S (STOP)
🔚 Session ended.


In [1]:
import cv2
import mediapipe as mp
import numpy as np
from collections import deque
import time
import socket
import requests
import threading

# =====================================================
#   EYEonic — Gaze-Controlled System v3.0
#   Python Eye Tracking Controller
# =====================================================

# ===== WiFi =====
ESP32_IP   = "192.168.4.1"
ESP32_PORT = 5005
sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

# ===== Telegram =====
BOT_TOKEN = "8988670376:AAH0b2ULG0Bq3mbjf0qlX6pEmLfIS2xyXmY"
CHAT_IDS  = ["1402303562"]

def send_telegram(msg):
    def _send():
        url = f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage"
        for cid in CHAT_IDS:
            try:
                requests.post(url, data={
                    "chat_id": cid,
                    "text": msg,
                    "parse_mode": "HTML"
                }, timeout=5)
                print(f"✅ Telegram → {cid}")
            except Exception as e:
                print(f"❌ Telegram: {e}")
    threading.Thread(target=_send, daemon=True).start()

# ===== MediaPipe =====
mp_face_mesh = mp.solutions.face_mesh
face_mesh    = mp_face_mesh.FaceMesh(refine_landmarks=True, max_num_faces=1)
cap          = cv2.VideoCapture(0)
print("📷 Camera:", cap.isOpened())

# ===== Landmarks =====
LEFT_IRIS   = [474, 475, 476, 477]
RIGHT_IRIS  = [469, 470, 471, 472]
LEFT_EYE_H  = [33, 133]
RIGHT_EYE_H = [362, 263]
LEFT_EYE    = [33, 160, 158, 133, 153, 144]
RIGHT_EYE   = [362, 385, 387, 263, 373, 380]

# ===== Timing =====
FPS_EST           = 30
BLINK_THRESHOLD   = 0.20

CLOSE_SEC_TOGGLE  = 2
CLOSE_SEC_WARN    = 15
CLOSE_SEC_EMERG   = 20
RECOVERY_SEC      = 5

FRAMES_TOGGLE  = FPS_EST * CLOSE_SEC_TOGGLE   # 60
FRAMES_WARN    = FPS_EST * CLOSE_SEC_WARN     # 450
FRAMES_EMERG   = FPS_EST * CLOSE_SEC_EMERG    # 600
FRAMES_RECOVER = FPS_EST * RECOVERY_SEC       # 150

# ===== State =====
is_stopped     = False
emergencyMode  = False
toggleDone     = False
warnSent       = False
emergSent      = False
tgSent         = False
close_counter  = 0
open_counter   = 0

# ===== Smoothing =====
direction_history = deque(maxlen=10)
last_command      = None

# ===== Commands =====
COMMAND_MAP = {
    "FORWARD":   "F",
    "LEFT":      "L",
    "RIGHT":     "R",
    "STOP":      "S",
    "BLINKING":  "S",
    "EMERGENCY": "E",
    "WARNING":   "W",
    "RESUME":    "X",
}

# ===== FPS =====
frame_count = 0
start_time  = time.time()
fps         = 0

# =====================================================
# ================= Functions =========================
# =====================================================

def eye_aspect_ratio(eye):
    A = np.linalg.norm(np.array(eye[1]) - np.array(eye[5]))
    B = np.linalg.norm(np.array(eye[2]) - np.array(eye[4]))
    C = np.linalg.norm(np.array(eye[0]) - np.array(eye[3]))
    return (A + B) / (2.0 * C)

def get_iris_ratio(face, iris_ids, corner_ids):
    iris_x  = np.mean([face.landmark[i].x for i in iris_ids])
    x1      = face.landmark[corner_ids[0]].x
    x2      = face.landmark[corner_ids[1]].x
    return (iris_x - min(x1,x2)) / max(abs(x2-x1), 0.001)

def send_cmd(key):
    global last_command
    cmd = COMMAND_MAP.get(key)
    if cmd and cmd != last_command:
        try:
            sock.sendto(cmd.encode(), (ESP32_IP, ESP32_PORT))
            last_command = cmd
            print(f"📡 Sent: {cmd} ({key})")
        except Exception as e:
            print(f"❌ UDP: {e}")

def send_cmd_force(key):
    global last_command
    cmd = COMMAND_MAP.get(key)
    if cmd:
        try:
            sock.sendto(cmd.encode(), (ESP32_IP, ESP32_PORT))
            last_command = cmd
            print(f"📡 Force: {cmd} ({key})")
        except Exception as e:
            print(f"❌ UDP: {e}")

def smooth_direction(history):
    if not history: return "FORWARD"
    counts = {}
    for d in history:
        counts[d] = counts.get(d, 0) + 1
    return max(counts, key=counts.get)

def draw_panel(panel, direction, ear, ratio, fps_val,
               close_cnt, is_stp, emg_mode, open_cnt):

    h, pw = panel.shape[:2]

    COLORS = {
        "FORWARD":   (0,200,0),
        "LEFT":      (255,140,0),
        "RIGHT":     (255,140,0),
        "STOP":      (0,0,200),
        "BLINKING":  (180,0,180),
        "EMERGENCY": (0,0,255),
        "WARNING":   (0,165,255),
    }

    # Header
    cv2.rectangle(panel,(0,0),(pw,50),(20,20,20),-1)
    cv2.putText(panel,"EYEonic v3.0",(10,35),
                cv2.FONT_HERSHEY_DUPLEX,0.85,(0,255,255),2)

    # Status
    if emg_mode:   s_col,s_txt = (0,0,200),"EMERGENCY"
    elif is_stp:   s_col,s_txt = (0,0,160),"PAUSED"
    else:          s_col,s_txt = (0,160,0),"ACTIVE"

    cv2.rectangle(panel,(10,60),(pw-10,105),s_col,-1)
    cv2.putText(panel,s_txt,(20,93),cv2.FONT_HERSHEY_DUPLEX,1.0,(255,255,255),2)

    # Direction
    d_col = COLORS.get(direction,(80,80,80))
    cv2.rectangle(panel,(10,115),(pw-10,165),d_col,-1)
    cv2.putText(panel,direction,(15,153),cv2.FONT_HERSHEY_DUPLEX,0.85,(255,255,255),2)

    # Arrow
    cx,cy = pw//2, 215
    if direction == "FORWARD":
        cv2.arrowedLine(panel,(cx,cy+40),(cx,cy-40),(0,200,0),4,tipLength=0.4)
    elif direction == "LEFT":
        cv2.arrowedLine(panel,(cx+40,cy),(cx-40,cy),(255,140,0),4,tipLength=0.4)
    elif direction == "RIGHT":
        cv2.arrowedLine(panel,(cx-40,cy),(cx+40,cy),(255,140,0),4,tipLength=0.4)
    elif direction in ("STOP","EMERGENCY"):
        cv2.rectangle(panel,(cx-28,cy-28),(cx+28,cy+28),(0,0,200),-1)
        cv2.putText(panel,"||",(cx-16,cy+12),cv2.FONT_HERSHEY_SIMPLEX,1.2,(255,255,255),3)

    # Eye Close Bar
    by  = 270
    bw  = pw - 20
    prg = min(close_cnt / FRAMES_EMERG, 1.0)
    sec = min(close_cnt / FPS_EST, CLOSE_SEC_EMERG)
    bc  = (0,0,255) if emg_mode else \
          (0,165,255) if sec >= CLOSE_SEC_WARN else \
          (0,200,255)

    cv2.rectangle(panel,(10,by),(pw-10,by+14),(60,60,60),-1)
    cv2.rectangle(panel,(10,by),(10+int(bw*prg),by+14),bc,-1)
    cv2.rectangle(panel,(10,by),(pw-10,by+14),(255,255,255),1)

    # Mark at 15s
    mx = 10 + int(bw*(CLOSE_SEC_WARN/CLOSE_SEC_EMERG))
    cv2.line(panel,(mx,by),(mx,by+14),(255,255,255),2)
    cv2.putText(panel,f"Eye: {sec:.1f}s  15s=Warn  20s=SOS",
                (10,by+28),cv2.FONT_HERSHEY_SIMPLEX,0.38,(180,180,180),1)

    # Recovery bar
    if emg_mode and open_cnt > 0:
        ry   = by + 42
        rprg = min(open_cnt / FRAMES_RECOVER, 1.0)
        rsec = min(open_cnt / FPS_EST, RECOVERY_SEC)
        cv2.rectangle(panel,(10,ry),(pw-10,ry+12),(60,60,60),-1)
        cv2.rectangle(panel,(10,ry),(10+int(bw*rprg),ry+12),(0,200,0),-1)
        cv2.putText(panel,f"Recovery: {rsec:.1f}s / 5s",
                    (10,ry+26),cv2.FONT_HERSHEY_SIMPLEX,0.40,(0,200,0),1)

    # Stats
    cv2.putText(panel,"WiFi: ON",       (10,h-90),cv2.FONT_HERSHEY_SIMPLEX,0.55,(0,200,0),1)
    cv2.putText(panel,f"FPS:  {fps_val:.0f}",(10,h-68),cv2.FONT_HERSHEY_SIMPLEX,0.55,(200,200,200),1)
    cv2.putText(panel,f"EAR:  {ear:.2f}",   (10,h-46),cv2.FONT_HERSHEY_SIMPLEX,0.55,(0,255,255),1)
    cv2.putText(panel,f"Ratio:{ratio:.2f}", (10,h-24),cv2.FONT_HERSHEY_SIMPLEX,0.55,(0,255,255),1)

# =====================================================
# ================= Main Loop =========================
# =====================================================

while True:
    ret, frame = cap.read()
    if not ret: break

    frame = cv2.flip(frame, 1)
    rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb)
    h, w, _ = frame.shape

    frame_count += 1
    elapsed = time.time() - start_time
    if elapsed >= 1.0:
        fps = frame_count / elapsed
        frame_count = 0
        start_time  = time.time()

    direction = "FORWARD"

    # ===== No Face =====
    if not results.multi_face_landmarks:
        ov = frame.copy()
        cv2.rectangle(ov,(0,0),(w,h),(0,0,60),-1)
        cv2.addWeighted(ov,0.3,frame,0.7,0,frame)
        cv2.rectangle(frame,(w//2-200,h//2-40),(w//2+200,h//2+40),(0,0,180),-1)
        cv2.putText(frame,"NO FACE DETECTED",(w//2-185,h//2+12),
                    cv2.FONT_HERSHEY_DUPLEX,0.9,(255,255,255),2)
        send_cmd("STOP")
        cv2.imshow("EYEonic — Gaze Control", frame)
        if cv2.waitKey(1) & 0xFF == 27: break
        continue

    face = results.multi_face_landmarks[0]

    # ===== Eye Points =====
    left_eye, right_eye = [], []
    for idx in LEFT_EYE:
        left_eye.append((int(face.landmark[idx].x*w), int(face.landmark[idx].y*h)))
    for idx in RIGHT_EYE:
        right_eye.append((int(face.landmark[idx].x*w), int(face.landmark[idx].y*h)))

    ear = (eye_aspect_ratio(left_eye) + eye_aspect_ratio(right_eye)) / 2

    # =====================================================
    # ===== EYE CLOSE LOGIC ===============================
    # =====================================================

    if ear < BLINK_THRESHOLD:
        close_counter += 1
        open_counter   = 0  # Reset recovery

        # 20s → EMERGENCY
        if close_counter >= FRAMES_EMERG and not emergSent:
            emergSent     = True
            emergencyMode = True
            send_cmd_force("EMERGENCY")
            print("🚨 EMERGENCY MODE ON")

            if not tgSent:
                tgSent = True
                send_telegram(
                    "🚨 <b>EYEonic Emergency Alert</b>\n\n"
                )

        # 15s → WARNING
        elif close_counter >= FRAMES_WARN and not warnSent:
            warnSent = True
            send_cmd_force("WARNING")
            print("⚠️ WARNING: Are you still there?")

        # 2s → Toggle STOP/GO 
        elif close_counter >= FRAMES_TOGGLE and not toggleDone and not warnSent:
            toggleDone = True
            is_stopped = not is_stopped
            print(f"🚦 Control: {'STOP' if is_stopped else 'GO'}")

    else:
        # ===== Eye Open =====

        if emergencyMode:
            # Recovery mode
            open_counter += 1

            if open_counter >= FRAMES_RECOVER:
                emergencyMode = False
                emergSent     = False
                warnSent      = False
                toggleDone    = False
                tgSent        = False
                close_counter = 0
                open_counter  = 0
                send_cmd_force("RESUME")
                print("✅ Recovery Complete — Resumed")

        else:
            # Normal reset
            if close_counter > 0:
                close_counter = 0
                toggleDone    = False
                warnSent      = False
                emergSent     = False

    # ===== Iris Ratio =====
    ratio = (get_iris_ratio(face, LEFT_IRIS,  LEFT_EYE_H) +
             get_iris_ratio(face, RIGHT_IRIS, RIGHT_EYE_H)) / 2

    # ===== Direction =====
    if emergencyMode:
        raw_dir = "EMERGENCY"
    elif ear < BLINK_THRESHOLD:
        raw_dir = "BLINKING"
    elif is_stopped:
        raw_dir = "STOP"
    elif ratio < 0.40:
        raw_dir = "LEFT"
    elif ratio > 0.60:
        raw_dir = "RIGHT"
    else:
        raw_dir = "FORWARD"

    direction_history.append(raw_dir)
    direction = smooth_direction(direction_history)
    send_cmd(direction)

    # ===== Draw =====
    for idx in LEFT_IRIS + RIGHT_IRIS:
        cv2.circle(frame,
                   (int(face.landmark[idx].x*w), int(face.landmark[idx].y*h)),
                   3,(0,255,0),-1)
    for p in left_eye + right_eye:
        cv2.circle(frame, p, 1, (255,140,0), -1)

    panel = np.full((h, 320, 3), (30,30,30), dtype=np.uint8)
    draw_panel(panel, direction, ear, ratio, fps,
               close_counter, is_stopped, emergencyMode, open_counter)

    cv2.imshow("EYEonic — Gaze Control", np.hstack((frame, panel)))

    if cv2.waitKey(1) & 0xFF == 27: break

cap.release()
cv2.destroyAllWindows()
sock.close()
print("🔚 Session ended.")

C:\Users\ALIENWARE\anaconda3\envs\eye_env\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


📷 Camera: True
📡 Sent: F (FORWARD)
📡 Sent: L (LEFT)
📡 Sent: F (FORWARD)
📡 Sent: L (LEFT)
📡 Sent: F (FORWARD)
📡 Sent: S (BLINKING)
📡 Sent: F (FORWARD)
📡 Sent: L (LEFT)
📡 Sent: F (FORWARD)
📡 Sent: L (LEFT)
📡 Sent: F (FORWARD)
📡 Sent: L (LEFT)
📡 Sent: S (STOP)
📡 Sent: L (LEFT)
📡 Sent: S (STOP)
📡 Sent: L (LEFT)
📡 Sent: S (STOP)
📡 Sent: L (LEFT)
📡 Sent: S (STOP)
📡 Sent: L (LEFT)
📡 Sent: S (STOP)
📡 Sent: L (LEFT)
📡 Sent: S (STOP)
📡 Sent: L (LEFT)
📡 Sent: S (STOP)
📡 Sent: L (LEFT)
📡 Sent: S (STOP)
📡 Sent: L (LEFT)
📡 Sent: R (RIGHT)
📡 Sent: S (STOP)
📡 Sent: R (RIGHT)
📡 Sent: L (LEFT)
📡 Sent: F (FORWARD)
📡 Sent: L (LEFT)
📡 Sent: F (FORWARD)
📡 Sent: L (LEFT)
📡 Sent: F (FORWARD)
📡 Sent: L (LEFT)
📡 Sent: S (STOP)
📡 Sent: L (LEFT)
📡 Sent: S (STOP)
📡 Sent: L (LEFT)
📡 Sent: S (STOP)
📡 Sent: L (LEFT)
📡 Sent: S (BLINKING)
📡 Sent: L (LEFT)
📡 Sent: S (BLINKING)
📡 Sent: L (LEFT)
📡 Sent: F (FORWARD)
📡 Sent: R (RIGHT)
📡 Sent: L (LEFT)
📡 Sent: R (RIGHT)
📡 Sent: L (LEFT)
📡 Sent: S (BLINKING)
📡 Sent: L (LEFT)
